# EduMentor AI - Baseline ML Training

This notebook trains baseline classifiers for dropout risk prediction and saves the best model.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
DATA_PROCESSED = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

clean_path = DATA_PROCESSED / 'clean_data.csv'
if not clean_path.exists():
    raise FileNotFoundError('Run notebooks/preprocessing.ipynb first to create clean_data.csv')

df = pd.read_csv(clean_path)
df.shape

In [ ]:
target_col = 'risk_label'
if target_col not in df.columns:
    raise ValueError('risk_label column missing in clean_data.csv')

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_cols),
    ('categorical', categorical_pipeline, categorical_cols),
])

In [ ]:
models = {
    'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'random_forest': RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight='balanced', min_samples_leaf=2
    ),
}

results = []
best_name = None
best_f1 = -1
best_pipeline = None

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')

    y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else None
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    results.append({'model': name, 'f1_macro': f1_macro, 'roc_auc': roc_auc})

    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_name = name
        best_pipeline = pipeline

results_df = pd.DataFrame(results).sort_values(by='f1_macro', ascending=False)
results_df

In [ ]:
model_path = MODELS_DIR / 'risk_model.joblib'
report_path = MODELS_DIR / 'model_comparison.csv'

joblib.dump(best_pipeline, model_path)
results_df.to_csv(report_path, index=False)

print(f'Best model: {best_name} | F1-macro: {best_f1:.4f}')
print(f'Saved model: {model_path}')
print(f'Saved report: {report_path}')